# Callback Manager API

Core callback dispatchers and run managers used by LangChain to send lifecycle events to callback handlers.

This module provides:

* Synchronous and asynchronous callback managers.
* Bound managers for LLM, chain, tool, and retriever runs.
* Event-dispatch helpers for sync and async callback handlers.
* Context managers for grouping unrelated operations into one traced chain run.
* Configuration helpers for handlers, tags, metadata, debugging, and LangSmith tracing.
* Utilities for dispatching user-defined custom events.

# Callback Manager Hierarchy

```text
BaseCallbackManager
├── CallbackManager
│   └── CallbackManagerForChainGroup
└── AsyncCallbackManager
    └── AsyncCallbackManagerForChainGroup

BaseRunManager
├── RunManager
│   ├── ParentRunManager
│   │   ├── CallbackManagerForChainRun
│   │   ├── CallbackManagerForToolRun
│   │   └── CallbackManagerForRetrieverRun
│   └── CallbackManagerForLLMRun
└── AsyncRunManager
    ├── AsyncParentRunManager
    │   ├── AsyncCallbackManagerForChainRun
    │   ├── AsyncCallbackManagerForToolRun
    │   └── AsyncCallbackManagerForRetrieverRun
    └── AsyncCallbackManagerForLLMRun
```

# Event Dispatch Functions

## `handle_event`

Dispatches one callback event from synchronous code.

```python
handle_event(
    handlers: list[BaseCallbackHandler],
    event_name: str,
    ignore_condition_name: str | None,
    *args: Any,
    **kwargs: Any
) -> None
```

## Parameters

* `handlers` — Callback handlers that may receive the event.
* `event_name` — Handler method name, such as `"on_llm_start"`.
* `ignore_condition_name` — Handler property that skips the event when truthy.
  * Examples:
    * `"ignore_llm"`
    * `"ignore_chain"`
    * `"ignore_agent"`
    * `"ignore_retriever"`
    * `"ignore_retry"`
    * `"ignore_custom_event"`
  * `None` means no ignore property is checked.
* `*args` — Positional arguments forwarded to the callback method.
* `**kwargs` — Keyword arguments forwarded to the callback method.

## Behaviour

For every handler:

1. Checks the configured ignore property.
2. Calls the corresponding handler method.
3. Collects returned coroutines from async handlers.
4. Logs callback exceptions.
5. Re-raises synchronous callback errors when:
   ```python
   handler.raise_error is True
   ```

When async handlers are called from this synchronous path:

* Their coroutines are executed after all handlers have been inspected.
* If an event loop is already running, execution is moved to a shared background thread.
* Otherwise, the coroutines run directly through `_run_coros`.

### Chat-Model Fallback

When a handler raises `NotImplementedError` from:

```python
on_chat_model_start
```

the dispatcher converts each message list into a prompt string using:

```python
get_buffer_string
```

and calls:

```python
on_llm_start
```

for that handler.

The same fallback is preserved for async handlers whose coroutine is collected by the synchronous dispatcher.

### Error Asymmetry

Synchronous callback exceptions honor `handler.raise_error`.

Exceptions raised by async callback coroutines executed through `_run_coros` are logged and swallowed, even when `handler.raise_error=True`.

## `ahandle_event`

Asynchronously dispatches one callback event.

```python
async def ahandle_event(
    handlers: list[BaseCallbackHandler],
    event_name: str,
    ignore_condition_name: str | None,
    *args: Any,
    **kwargs: Any
) -> None
```

## Behaviour

1. Executes handlers with `run_inline=True` sequentially.
2. Executes remaining handlers concurrently with `asyncio.gather`.
3. Awaits native async callback methods directly.
4. Runs synchronous non-inline callback methods in an executor.
5. Preserves the current `contextvars` context for executor-based calls.
6. Applies the same `on_chat_model_start` to `on_llm_start` fallback.
7. Re-raises callback errors when `handler.raise_error=True`.

## `_ahandle_event_for_handler`

Executes one callback method in the asynchronous dispatch path.

```python
async def _ahandle_event_for_handler(
    handler: BaseCallbackHandler,
    event_name: str,
    ignore_condition_name: str | None,
    *args: Any,
    **kwargs: Any
) -> None
```

It selects one execution mode:

```text
Async callback method
    -> await directly

Sync callback with run_inline=True
    -> call directly on the event-loop thread

Sync callback with run_inline=False
    -> run in executor
```

## `_achat_model_start_fallback`

Wraps a collected async `on_chat_model_start` coroutine.

```python
async def _achat_model_start_fallback(
    coro: Coroutine[Any, Any, Any],
    handler: BaseCallbackHandler,
    *args: Any,
    **kwargs: Any
) -> None
```

If the coroutine raises `NotImplementedError`, it invokes the handler's async or synchronous `on_llm_start` callback using stringified messages.

## `_run_coros`

Runs callback coroutines from synchronous code.

```python
_run_coros(
    coros: list[
        Coroutine[Any, Any, Any]
    ]
) -> None
```

### Python 3.11 and Later

Uses:

```python
asyncio.Runner()
```

It:

* Runs each callback coroutine.
* Logs exceptions.
* Continues running pending tasks scheduled by callback coroutines.
* Closes asynchronous generators and executors with the runner.

### Earlier Python Versions

Runs each coroutine separately using:

```python
asyncio.run(coro)
```

# `shielded`

Decorator that protects an async callback method from caller cancellation.

```python
shielded(
    func: Func
) -> Func
```

## Behaviour

The wrapper:

1. Copies the current `contextvars` context.
2. Creates a task for the callback coroutine.
3. Uses:
   ```python
   asyncio.shield(task)
   ```
4. On Python 3.11+, passes the copied context explicitly to `asyncio.create_task`.
5. Falls back to normal task creation when the `context` parameter is unavailable.

The decorator is used for selected terminal callbacks where completion or error reporting should continue even if the caller is cancelled.

At this revision, shielded methods include:

* `AsyncCallbackManagerForLLMRun.on_llm_end`
* `AsyncCallbackManagerForLLMRun.on_llm_error`
* `AsyncCallbackManagerForChainRun.on_chain_end`
* `AsyncCallbackManagerForChainRun.on_chain_error`
* `AsyncCallbackManagerForRetrieverRun.on_retriever_end`
* `AsyncCallbackManagerForRetrieverRun.on_retriever_error`

# `BaseRunManager`

Base class for a callback manager bound to one specific run.

- Bases: `RunManagerMixin`

## Constructor

```python
BaseRunManager(
    *,
    run_id: UUID,
    handlers: list[BaseCallbackHandler],
    inheritable_handlers: list[BaseCallbackHandler],
    parent_run_id: UUID | None = None,
    tags: list[str] | None = None,
    inheritable_tags: list[str] | None = None,
    metadata: dict[str, Any] | None = None,
    inheritable_metadata: dict[str, Any] | None = None
)
```

## Attributes

* `run_id` — Identifier of the current run.
* `handlers` — All handlers active for this run.
* `inheritable_handlers` — Handlers propagated to child runs.
* `parent_run_id` — Identifier of the parent run.
* `tags` — All tags active for this run.
* `inheritable_tags` — Tags propagated to child runs.
* `metadata` — All metadata active for this run.
* `inheritable_metadata` — Metadata propagated to child runs.

Missing tag lists and metadata dictionaries are normalized to empty containers.

## `get_noop_manager`

Creates a run manager with no handlers.

```python
@classmethod
def get_noop_manager(
    cls
) -> Self
```

The manager receives:

* A new UUIDv7 run ID.
* Empty handler lists.
* Empty tags.
* Empty metadata.

Calls on the resulting manager perform no callback work.

# `RunManager`

Synchronous base manager for events occurring during an active run.

- Bases: `BaseRunManager`

## Methods

### `on_text`

Dispatches arbitrary text.

```python
on_text(
    self,
    text: str,
    **kwargs: Any
) -> None
```

It calls each handler's:

```python
on_text
```

with the bound run ID, parent run ID, and tags.

### `on_retry`

Dispatches a Tenacity retry event.

```python
on_retry(
    self,
    retry_state: RetryCallState,
    **kwargs: Any
) -> None
```

Handlers with `ignore_retry=True` are skipped.

# `ParentRunManager`

Synchronous run manager capable of creating callback managers for child runs.

- Bases: `RunManager`

## `get_child`

Creates a `CallbackManager` whose parent is the current run.

```python
get_child(
    self,
    tag: str | None = None
) -> CallbackManager
```

The child manager receives only inheritable values from the parent:

* `inheritable_handlers`
* `inheritable_tags`
* `inheritable_metadata`

An optional `tag` is added locally to the child and is not inherited further through that addition.

# `AsyncRunManager`

Abstract asynchronous run manager.

- Bases:
  * `BaseRunManager`
  * `ABC`

## `get_sync`

Returns a synchronous manager representing the same run.

```python
@abstractmethod
def get_sync(
    self
) -> RunManager
```

Concrete async run managers override this with a more specific return type.

## `on_text`

```python
async def on_text(
    self,
    text: str,
    **kwargs: Any
) -> None
```

Uses `ahandle_event`.

## `on_retry`

```python
async def on_retry(
    self,
    retry_state: RetryCallState,
    **kwargs: Any
) -> None
```

Uses `ahandle_event` and respects `ignore_retry`.

# `AsyncParentRunManager`

Asynchronous run manager capable of creating child callback managers.

- Bases: `AsyncRunManager`

## `get_child`

```python
get_child(
    self,
    tag: str | None = None
) -> AsyncCallbackManager
```

The child inherits handlers, tags, and metadata in the same way as `ParentRunManager.get_child`.

# `CallbackManagerForLLMRun`

Bound synchronous callback manager for one LLM or chat-model run.

- Bases:
  * `RunManager`
  * `LLMManagerMixin`

## Methods

### `on_llm_new_token`

```python
on_llm_new_token(
    self,
    token: str | list[
        str | dict[str, Any]
    ],
    *,
    chunk: (
        GenerationChunk
        | ChatGenerationChunk
        | None
    ) = None,
    **kwargs: Any
) -> None
```

Dispatches streaming tokens or content blocks.

Handlers with `ignore_llm=True` are skipped.

### `on_llm_end`

```python
on_llm_end(
    self,
    response: LLMResult,
    **kwargs: Any
) -> None
```

Dispatches successful model completion.

### `on_llm_error`

```python
on_llm_error(
    self,
    error: BaseException,
    **kwargs: Any
) -> None
```

Optional keyword arguments may include a partially generated `LLMResult`.

### `on_stream_event`

```python
on_stream_event(
    self,
    event: MessagesData,
    **kwargs: Any
) -> None
```

Dispatches protocol events produced by:

```python
stream_events(version="v3")
```

# `AsyncCallbackManagerForLLMRun`

Bound asynchronous callback manager for one model run.

- Bases:
  * `AsyncRunManager`
  * `LLMManagerMixin`

## `get_sync`

```python
get_sync(
    self
) -> CallbackManagerForLLMRun
```

Creates a synchronous manager using the same run identifiers, handlers, tags, and metadata.

## Async Methods

```python
async def on_llm_new_token(...)
async def on_llm_end(...)
async def on_llm_error(...)
async def on_stream_event(...)
```

`on_llm_end` and `on_llm_error` are cancellation-shielded.

# `CallbackManagerForChainRun`

Bound synchronous manager for one chain or agent run.

- Bases:
  * `ParentRunManager`
  * `ChainManagerMixin`

## Methods

### `on_chain_end`

```python
on_chain_end(
    self,
    outputs: dict[str, Any] | Any,
    **kwargs: Any
) -> None
```

### `on_chain_error`

```python
on_chain_error(
    self,
    error: BaseException,
    **kwargs: Any
) -> None
```

### `on_agent_action`

```python
on_agent_action(
    self,
    action: AgentAction,
    **kwargs: Any
) -> None
```

### `on_agent_finish`

```python
on_agent_finish(
    self,
    finish: AgentFinish,
    **kwargs: Any
) -> None
```

Chain events respect `ignore_chain`; agent events respect `ignore_agent`.

# `AsyncCallbackManagerForChainRun`

Bound asynchronous manager for one chain or agent run.

- Bases:
  * `AsyncParentRunManager`
  * `ChainManagerMixin`

## Methods

```python
get_sync(
    self
) -> CallbackManagerForChainRun
```

```python
async def on_chain_end(...)
async def on_chain_error(...)
async def on_agent_action(...)
async def on_agent_finish(...)
```

`on_chain_end` and `on_chain_error` are cancellation-shielded.

# `CallbackManagerForToolRun`

Bound synchronous manager for one tool run.

- Bases:
  * `ParentRunManager`
  * `ToolManagerMixin`

## Methods

### `on_tool_end`

```python
on_tool_end(
    self,
    output: Any,
    **kwargs: Any
) -> None
```

### `on_tool_error`

```python
on_tool_error(
    self,
    error: BaseException,
    **kwargs: Any
) -> None
```

Both events use the handler's `ignore_agent` flag.

# `AsyncCallbackManagerForToolRun`

Bound asynchronous manager for one tool run.

- Bases:
  * `AsyncParentRunManager`
  * `ToolManagerMixin`

## Methods

```python
get_sync(
    self
) -> CallbackManagerForToolRun
```

```python
async def on_tool_end(...)
async def on_tool_error(...)
```

# `CallbackManagerForRetrieverRun`

Bound synchronous manager for one retriever run.

- Bases:
  * `ParentRunManager`
  * `RetrieverManagerMixin`

## Methods

### `on_retriever_end`

```python
on_retriever_end(
    self,
    documents: Sequence[Document],
    **kwargs: Any
) -> None
```

### `on_retriever_error`

```python
on_retriever_error(
    self,
    error: BaseException,
    **kwargs: Any
) -> None
```

Both respect `ignore_retriever`.

# `AsyncCallbackManagerForRetrieverRun`

Bound asynchronous manager for one retriever run.

- Bases:
  * `AsyncParentRunManager`
  * `RetrieverManagerMixin`

## Methods

```python
get_sync(
    self
) -> CallbackManagerForRetrieverRun
```

```python
async def on_retriever_end(...)
async def on_retriever_error(...)
```

Both terminal methods are cancellation-shielded.

# `CallbackManager`

Root synchronous callback manager used to start new runs.

- Bases: `BaseCallbackManager`

## Inherited Constructor

```python
CallbackManager(
    handlers: list[BaseCallbackHandler],
    inheritable_handlers: (
        list[BaseCallbackHandler]
        | None
    ) = None,
    parent_run_id: UUID | None = None,
    *,
    tags: list[str] | None = None,
    inheritable_tags: list[str] | None = None,
    metadata: dict[str, Any] | None = None,
    inheritable_metadata: (
        dict[str, Any]
        | None
    ) = None
)
```

## Methods

### `on_llm_start`

Starts one LLM run per prompt.

```python
on_llm_start(
    self,
    serialized: dict[str, Any],
    prompts: list[str],
    run_id: UUID | None = None,
    **kwargs: Any
) -> list[
    CallbackManagerForLLMRun
]
```

For multiple prompts:

* The explicitly supplied `run_id` is used only for the first prompt.
* Later prompts receive new UUIDv7 identifiers.
* One bound run manager is returned per prompt.

### `on_chat_model_start`

Starts one model run per message batch.

```python
on_chat_model_start(
    self,
    serialized: dict[str, Any],
    messages: list[
        list[BaseMessage]
    ],
    run_id: UUID | None = None,
    **kwargs: Any
) -> list[
    CallbackManagerForLLMRun
]
```

The first message batch may use the supplied run ID. Remaining batches receive new IDs.

### `on_chain_start`

```python
on_chain_start(
    self,
    serialized: dict[str, Any] | None,
    inputs: dict[str, Any] | Any,
    run_id: UUID | None = None,
    **kwargs: Any
) -> CallbackManagerForChainRun
```

Creates a UUIDv7 run ID when one is not supplied.

### `on_tool_start`

```python
on_tool_start(
    self,
    serialized: dict[str, Any] | None,
    input_str: str,
    run_id: UUID | None = None,
    parent_run_id: UUID | None = None,
    inputs: dict[str, Any] | None = None,
    **kwargs: Any
) -> CallbackManagerForToolRun
```

`inputs` may preserve the original dictionary arguments when `input_str` is only a string representation.

### `on_retriever_start`

```python
on_retriever_start(
    self,
    serialized: dict[str, Any] | None,
    query: str,
    run_id: UUID | None = None,
    parent_run_id: UUID | None = None,
    **kwargs: Any
) -> CallbackManagerForRetrieverRun
```

### `on_custom_event`

Dispatches a user-defined event.

```python
on_custom_event(
    self,
    name: str,
    data: Any,
    run_id: UUID | None = None,
    **kwargs: Any
) -> None
```

The public dispatcher does not accept arbitrary extra keyword arguments.

Additional values should be included inside `data`.

It raises `ValueError` when `kwargs` is non-empty.

### `configure`

Creates a callback manager from local and inheritable callback configuration.

```python
@classmethod
def configure(
    cls,
    inheritable_callbacks: Callbacks = None,
    local_callbacks: Callbacks = None,
    verbose: bool = False,
    inheritable_tags: (
        list[str] | None
    ) = None,
    local_tags: (
        list[str] | None
    ) = None,
    inheritable_metadata: (
        dict[str, Any] | None
    ) = None,
    local_metadata: (
        dict[str, Any] | None
    ) = None,
    *,
    langsmith_inheritable_metadata: (
        Mapping[str, Any] | None
    ) = None,
    langsmith_inheritable_tags: (
        list[str] | None
    ) = None
) -> CallbackManager
```

It delegates to `_configure`.

# `AsyncCallbackManager`

Root asynchronous callback manager.

- Bases: `BaseCallbackManager`

## Property: `is_async`

```python
@property
def is_async(
    self
) -> bool
```

Always returns:

```python
True
```

## Async Start Methods

```python
async def on_llm_start(...)
async def on_chat_model_start(...)
async def on_chain_start(...)
async def on_tool_start(...)
async def on_retriever_start(...)
async def on_custom_event(...)
```

They return async bound run managers corresponding to the synchronous variants.

Inline callbacks are awaited sequentially, while non-inline callbacks are executed through the async event dispatcher.

## `configure`

```python
@classmethod
def configure(
    cls,
    inheritable_callbacks: Callbacks = None,
    local_callbacks: Callbacks = None,
    verbose: bool = False,
    inheritable_tags: (
        list[str] | None
    ) = None,
    local_tags: (
        list[str] | None
    ) = None,
    inheritable_metadata: (
        dict[str, Any] | None
    ) = None,
    local_metadata: (
        dict[str, Any] | None
    ) = None,
    *,
    langsmith_inheritable_metadata: (
        Mapping[str, Any] | None
    ) = None,
    langsmith_inheritable_tags: (
        list[str] | None
    ) = None
) -> AsyncCallbackManager
```

# Chain-Group Context Managers

## `trace_as_chain_group`

Groups several synchronous operations under one traced chain run.

```python
@contextmanager
def trace_as_chain_group(
    group_name: str,
    callback_manager: (
        CallbackManager | None
    ) = None,
    *,
    inputs: (
        dict[str, Any] | None
    ) = None,
    project_name: str | None = None,
    example_id: (
        str | UUID | None
    ) = None,
    run_id: UUID | None = None,
    tags: list[str] | None = None,
    metadata: (
        dict[str, Any] | None
    ) = None
) -> Generator[
    CallbackManagerForChainGroup,
    None,
    None
]
```

## Behaviour

1. Resolves tracing callbacks.
2. Configures a `CallbackManager`.
3. Starts one chain run named `group_name`.
4. Creates a child callback manager for work inside the group.
5. Yields a `CallbackManagerForChainGroup`.
6. Automatically reports:
   * `on_chain_error` when the block raises.
   * `on_chain_end({})` when the block finishes normally.
7. Skips automatic completion when the group manager has already been ended manually.

LangSmith visibility requires tracing to be enabled, commonly through:

```text
LANGCHAIN_TRACING_V2=true
```

## Example

```python
from langchain_core.callbacks.manager import (
    trace_as_chain_group,
)

with trace_as_chain_group(
    "research_workflow",
    inputs={
        "topic": "LangChain callbacks"
    },
) as manager:
    result = runnable.invoke(
        {
            "topic": "LangChain callbacks"
        },
        {
            "callbacks": manager
        },
    )

    manager.on_chain_end(
        {
            "output": result
        }
    )
```

## `atrace_as_chain_group`

Asynchronous equivalent of `trace_as_chain_group`.

```python
@asynccontextmanager
async def atrace_as_chain_group(
    group_name: str,
    callback_manager: (
        AsyncCallbackManager | None
    ) = None,
    *,
    inputs: (
        dict[str, Any] | None
    ) = None,
    project_name: str | None = None,
    example_id: (
        str | UUID | None
    ) = None,
    run_id: UUID | None = None,
    tags: list[str] | None = None,
    metadata: (
        dict[str, Any] | None
    ) = None
) -> AsyncGenerator[
    AsyncCallbackManagerForChainGroup,
    None
]
```

# `CallbackManagerForChainGroup`

Synchronous callback manager representing an open traced chain group.

- Bases: `CallbackManager`

## Constructor

```python
CallbackManagerForChainGroup(
    handlers: list[
        BaseCallbackHandler
    ],
    inheritable_handlers: (
        list[BaseCallbackHandler]
        | None
    ) = None,
    parent_run_id: UUID | None = None,
    *,
    parent_run_manager: (
        CallbackManagerForChainRun
    ),
    **kwargs: Any
)
```

## Additional Attributes

* `parent_run_manager` — Bound manager for the chain-group run.
* `ended` — Whether the group has already received an end or error event.

## Methods

### `copy`

```python
copy(
    self
) -> CallbackManagerForChainGroup
```

Copies handlers, tags, metadata, and the same parent run manager.

### `merge`

```python
merge(
    self,
    other: BaseCallbackManager
) -> CallbackManagerForChainGroup
```

Merges:

* Handlers.
* Inheritable handlers.
* Tags.
* Inheritable tags.
* Metadata.

The current group's `parent_run_manager` is preserved.

Metadata from `other` overwrites duplicate keys from the current manager because of dictionary merge order.

### `on_chain_end`

```python
on_chain_end(
    self,
    outputs: dict[str, Any] | Any,
    **kwargs: Any
) -> None
```

Sets:

```python
ended = True
```

and delegates to the parent run manager.

### `on_chain_error`

```python
on_chain_error(
    self,
    error: BaseException,
    **kwargs: Any
) -> None
```

Also marks the group as ended before delegation.

# `AsyncCallbackManagerForChainGroup`

Asynchronous chain-group manager.

- Bases: `AsyncCallbackManager`

It provides async versions of:

```python
on_chain_end
on_chain_error
```

and synchronous:

```python
copy
merge
```

It preserves the associated `AsyncCallbackManagerForChainRun`.

# Callback Configuration

## `_configure`

Shared implementation used by both root callback manager classes.

```python
_configure(
    callback_manager_cls: type[T],
    inheritable_callbacks: Callbacks = None,
    local_callbacks: Callbacks = None,
    inheritable_tags: (
        list[str] | None
    ) = None,
    local_tags: (
        list[str] | None
    ) = None,
    inheritable_metadata: (
        dict[str, Any] | None
    ) = None,
    local_metadata: (
        dict[str, Any] | None
    ) = None,
    *,
    verbose: bool = False,
    langsmith_inheritable_metadata: (
        Mapping[str, Any] | None
    ) = None,
    langsmith_inheritable_tags: (
        list[str] | None
    ) = None
) -> T
```

## Inheritable and Local Values

### Inheritable

Values passed as inheritable are copied into child callback managers.

```text
inheritable_callbacks
inheritable_tags
inheritable_metadata
```

### Local

Local values are active only on the configured manager.

```text
local_callbacks
local_tags
local_metadata
```

Local handlers are added with:

```python
inherit=False
```

## Existing Callback Manager Input

`inheritable_callbacks` may be:

* A list of handlers.
* An existing callback manager.
* `None`.

When an existing manager is supplied, its:

* Handlers.
* Inheritable handlers.
* Tags.
* Inheritable tags.
* Metadata.
* Inheritable metadata.
* Parent run ID.

are copied.

The helper reconciles the inherited parent run ID with the active LangSmith tracing context.

## Verbose and Debug Handlers

When `verbose=True`, the helper adds:

```python
StdOutCallbackHandler
```

unless one is already present.

When global debug mode is enabled, it adds:

```python
ConsoleCallbackHandler
```

unless one is already present.

## LangSmith Tracing

When tracing v2 is enabled, the helper adds a `LangChainTracer` unless one is already present.

It may use:

* A tracer stored in a context variable.
* A newly created tracer.
* The client and parent run from the current tracing context.
* The configured tracer project.
* Tracing tags and metadata.

If tracer creation fails, it logs a warning rather than stopping callback configuration.

## Legacy Tracing Error

When either legacy variable is set:

```text
LANGCHAIN_TRACING
LANGCHAIN_HANDLER
```

without tracing v2 being enabled, `_configure` raises `RuntimeError`.

The error instructs the user to enable:

```text
LANGCHAIN_TRACING_V2
```

## Hook-Configured Handlers

The helper processes callback-handler hooks registered through the tracing context.

Handlers may be:

* Retrieved from context variables.
* Created from registered handler classes when a corresponding environment variable is enabled.
* Added as inheritable or local according to hook configuration.

## LangSmith Default Metadata and Tags

When supplied, `langsmith_inheritable_metadata` and `langsmith_inheritable_tags` are applied to each `LangChainTracer` using:

```python
copy_with_metadata_defaults
```

Tracing-context metadata is merged before caller-supplied defaults, allowing explicitly supplied keys to take precedence.

# Custom Event Utilities

## `dispatch_custom_event`

Dispatches a custom event from synchronous runnable code.

```python
dispatch_custom_event(
    name: str,
    data: Any,
    *,
    config: RunnableConfig | None = None
) -> None
```

## Parameters

* `name` — User-defined event name.
* `data` — Free-form event payload.
  * JSON-serializable data is recommended but not enforced.
* `config` — Optional runnable configuration.

## Parent-Run Requirement

The helper obtains the callback manager from the current runnable config and associates the event with:

```python
callback_manager.parent_run_id
```

It raises `RuntimeError` when no parent run ID is available.

Therefore, it is intended to be called inside an existing run, such as:

* A tool.
* `RunnableLambda`.
* `RunnableGenerator`.

## Example

```python
from langchain_core.callbacks import (
    BaseCallbackHandler,
    dispatch_custom_event,
)
from langchain_core.runnables import (
    RunnableLambda,
)

class ProgressHandler(
    BaseCallbackHandler
):
    def on_custom_event(
        self,
        name,
        data,
        *,
        run_id,
        **kwargs,
    ):
        print(name, data)

def work(value):
    dispatch_custom_event(
        "progress",
        {
            "stage": "started"
        },
    )
    return value

runnable = RunnableLambda(work)

runnable.invoke(
    "input",
    {
        "callbacks": [
            ProgressHandler()
        ]
    },
)
```

## `adispatch_custom_event`

Asynchronous equivalent.

```python
async def adispatch_custom_event(
    name: str,
    data: Any,
    *,
    config: RunnableConfig | None = None
) -> None
```

It uses the async callback manager and awaits `on_custom_event`.

### Python 3.10 Note

In async code on Python 3.10, the runnable `config` may need to be supplied explicitly because automatic context propagation is limited.

# Shared Callback Executor

## `_executor`

Returns a lazily created global thread pool.

```python
@functools.lru_cache(
    maxsize=1
)
def _executor(
) -> ThreadPoolExecutor
```

Configuration:

```python
ThreadPoolExecutor(
    max_workers=10
)
```

The executor is used when synchronous event dispatch encounters async handlers while an event loop is already running.

It is shut down at process exit through:

```python
atexit.register(
    executor.shutdown,
    wait=True
)
```

# Common Event Flow

```text
Root CallbackManager
        |
        | on_*_start(...)
        v
Bound Run Manager
        |
        | on_*_new_token / on_text / on_retry
        | on_*_end / on_*_error
        v
Callback Handlers
```

## Child Run Flow

```text
ParentRunManager
        |
        | get_child()
        v
Child CallbackManager
        |
        | inherits only inheritable handlers,
        | tags, and metadata
        v
Nested run
```

# Example: Complete Synchronous Lifecycle

```python
from langchain_core.callbacks import (
    BaseCallbackHandler,
)
from langchain_core.callbacks.manager import (
    CallbackManager,
)

class Handler(
    BaseCallbackHandler
):
    def on_chain_start(
        self,
        serialized,
        inputs,
        **kwargs,
    ):
        print(
            "started",
            inputs,
        )

    def on_chain_end(
        self,
        outputs,
        **kwargs,
    ):
        print(
            "finished",
            outputs,
        )

manager = CallbackManager.configure(
    inheritable_callbacks=[
        Handler()
    ],
    inheritable_tags=[
        "demo"
    ],
)

run_manager = manager.on_chain_start(
    {
        "name": "example"
    },
    {
        "input": "hello"
    },
)

run_manager.on_text(
    "working..."
)

run_manager.on_chain_end(
    {
        "output": "done"
    }
)
```

# Example: Async Lifecycle

```python
from langchain_core.callbacks import (
    AsyncCallbackHandler,
)
from langchain_core.callbacks.manager import (
    AsyncCallbackManager,
)

class Handler(
    AsyncCallbackHandler
):
    async def on_llm_new_token(
        self,
        token,
        **kwargs,
    ):
        print(token)

manager = (
    AsyncCallbackManager
    .configure(
        inheritable_callbacks=[
            Handler()
        ]
    )
)

runs = await manager.on_llm_start(
    {
        "name": "model"
    },
    [
        "Explain callbacks."
    ],
)

await runs[0].on_llm_new_token(
    "Callbacks"
)
```

# Error Handling Summary

| Situation | Behaviour |
|---|---|
| Handler ignore property is true | Handler is skipped |
| Sync handler raises and `raise_error=False` | Warning is logged |
| Sync handler raises and `raise_error=True` | Exception is re-raised |
| Async handler raises in async dispatch and `raise_error=True` | Exception is re-raised |
| Async coroutine fails through sync `handle_event` | Warning is logged and failure is swallowed |
| `on_chat_model_start` raises `NotImplementedError` | Falls back to `on_llm_start` |
| Custom manager event receives extra kwargs | Raises `ValueError` |
| Custom-event utility has no parent run | Raises `RuntimeError` |
| Legacy tracing enabled without tracing v2 | Raises `RuntimeError` |

# Source

This reference follows the pinned LangChain source:

```text
libs/core/langchain_core/callbacks/manager.py
Commit: 1c3a4186cf2ba4f28face59118ac7786de009f91
```

Source URL:

```text
https://github.com/langchain-ai/langchain/blob/1c3a4186cf2ba4f28face59118ac7786de009f91/libs/core/langchain_core/callbacks/manager.py
```